# 【燧原】趣味英文完形填空闯关应用

基于 ERNIE-4.5-0.3B 轻量模型的趣味英文完形填空闯关应用，将 AI 动态出题、限时机制与积分系统相结合。

---

## 0. 配置环境

In [4]:
# 在【燧原】环境中需要安装特定版本的 paddlepaddle
# 参考：https://github.com/PaddlePaddle/community/blob/master/pfcc/paddle-hardware/%E7%87%A7%E5%8E%9F%E7%A7%91%E6%8A%80-%E5%9F%BA%E4%BA%8EFastDeploy%E8%B7%91%E9%80%9AERNIE-4.5-0.3B-Paddle%20%E6%89%93%E5%8D%A1%E4%BB%BB%E5%8A%A1.md
# # PaddlePaddle『飞桨』深度学习框架，提供运算基础能力
!python -m pip install paddlepaddle==3.1.0a0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

# # PaddleCustomDevice是PaddlePaddle『飞桨』深度学习框架的自定义硬件接入实现，提供GCU的算子实现
!python -m pip install paddle-custom-gcu==3.0.0.dev20250716 -i https://www.paddlepaddle.org.cn/packages/nightly/gcu/

!python -c "import paddle_custom_device; paddle_custom_device.gcu.version()"

!python -c "import paddle; paddle.utils.run_check()"

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cpu/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.1/195.1 MB 11.2 MB/s  0:00:17:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [paddlepaddle] [paddlepaddle]

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://www.paddlepaddle.org.cn/packages/nightly/gcu/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 49.3 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
version: 3.0.0.dev20250716
commit: 2bd3a8283dc9a91f537aa1377b7ed927aa08326d
TopsPlatform: 1.5.0.601
/usr/local/lib/python3.10/dist-packages/paddle/utils/cpp_extension/extension_utils.py:715: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccach

In [7]:
# 安装 FastDeploy

!python -m pip install --upgrade pip setuptools wheel

!python -m pip install -r requirements-gcu.txt --extra-index-url https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simplels
!python -m pip install fastdeploy -i https://www.paddlepaddle.org.cn/packages/stable/gcu/ --extra-index-url https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simplels

# 下载 ERNIE-4.5-0.3B-Paddle 模型

!huggingface-cli download baidu/ERNIE-4.5-0.3B-Paddle --local-dir baidu/ERNIE-4.5-0.3B-Paddle

!export ENABLE_V1_KVCACHE_SCHEDULER=1

# 下面这个环境变量主要是为了绕过单卡的一个小bug。
!export CUDA_VISIBLE_DEVICES=0


Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.3 MB/s  0:00:006m-:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.6 MB/s  0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.45.1
    Uninstalling wheel-0.45.1:
      Successfully uninstalled wheel-0.45.1
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.9.0
    Uninstalling setuptools-80.9.0:
      Successfully uninstalled setuptools-80.9.0━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
  Attempting uninstall: pip90m╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [setuptools]
    Found existing installation: pip 25.21m╸━━━━━━━━━━━━━ 2/3 [pip]ls]
    Uninstalling pip-25.2:━━━━╸━━━━━━━━━━━━━ 2/3 [pip]
      Successfully uninstalled pip-25.2╸━━━━━━━━━━━━━ 2/3 [pip]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pip]2/3 [pip]
ERROR: pip's dependency resolver does not currently take into account all the pac

## 1. 导入依赖

In [1]:
import re
import time
import openai


## 2. 加载模型

### 2.1 启动 FastDeploy OpenAI API Server

需要先启动 FastDeploy 的 OpenAI 兼容 API Server。该服务通过子进程启动，提供与 OpenAI 兼容的 HTTP 接口。

In [2]:
# ========== 启动 FastDeploy OpenAI API Server ==========
# 通过子进程启动 fastdeploy 的 openai api_server。
# 运行本 Cell 会：
#   1. 检查端口是否已被占用，若占用则杀死占用进程
#   2. 启动 fastdeploy 服务

import subprocess
import os
import time
import socket
import signal

# FastDeploy OpenAI API Server 配置
FD_MODEL = "baidu/ERNIE-4.5-0.3B-Paddle"
FD_HOST = "0.0.0.0"
FD_PORT = 8180
FD_METRICS_PORT = 8181
FD_WORKER_QUEUE_PORT = 8182
FD_MAX_MODEL_LEN = 32768
FD_MAX_NUM_SEQS = 32
FD_NUM_GPU_BLOCKS_OVERRIDE = 4896

# ---------- Step 1: 检查并释放被占用的端口 ----------
def is_port_in_use(port):
    """检查指定端口是否被占用"""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

def kill_process_on_port(port):
    """查找并杀死占用指定端口的进程"""
    try:
        result = subprocess.run(
            ["lsof", "-t", "-i", f":{port}"],
            capture_output=True, text=True
        )
        pids = result.stdout.strip().split('\n')
        for pid_str in pids:
            pid_str = pid_str.strip()
            if pid_str:
                pid = int(pid_str)
                print(f"端口 {port} 被进程 PID={pid} 占用，正在终止...")
                os.kill(pid, signal.SIGKILL)
                time.sleep(1)
                print(f"进程 PID={pid} 已终止。")
    except (subprocess.CalledProcessError, FileNotFoundError, ValueError, ProcessLookupError):
        try:
            subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)
            print(f"已通过 fuser 终止占用端口 {port} 的进程。")
        except FileNotFoundError:
            print(f"警告：无法终止占用端口 {port} 的进程（lsof/fuser 均不可用）。")

if is_port_in_use(FD_PORT):
    print(f"端口 {FD_PORT} 已被占用，需要先释放。")
    kill_process_on_port(FD_PORT)
    time.sleep(2)
    if is_port_in_use(FD_PORT):
        print(f"警告：端口 {FD_PORT} 仍被占用，启动可能失败！")
    else:
        print(f"端口 {FD_PORT} 已释放。")
else:
    print(f"端口 {FD_PORT} 未被占用，可以直接启动。")

# ---------- Step 2: 设置环境变量并启动服务 ----------
os.environ["ENABLE_V1_KVCACHE_SCHEDULER"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

fd_server_cmd = [
    "python", "-m", "fastdeploy.entrypoints.openai.api_server",
    "--model", FD_MODEL,
    "--host", FD_HOST,
    "--port", str(FD_PORT),
    "--metrics-port", str(FD_METRICS_PORT),
    "--engine-worker-queue-port", str(FD_WORKER_QUEUE_PORT),
    "--max-model-len", str(FD_MAX_MODEL_LEN),
    "--max-num-seqs", str(FD_MAX_NUM_SEQS),
    "--num-gpu-blocks-override", str(FD_NUM_GPU_BLOCKS_OVERRIDE),
]

fd_server_process = subprocess.Popen(
    fd_server_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print(f"FastDeploy OpenAI API Server 启动中... (PID: {fd_server_process.pid})")
print(f"服务地址: http://{FD_HOST}:{FD_PORT}/v1")
print("请等待服务就绪后再继续执行后续 Cell。")


端口 8180 未被占用，可以直接启动。
FastDeploy OpenAI API Server 启动中... (PID: 31668)
服务地址: http://0.0.0.0:8180/v1
请等待服务就绪后再继续执行后续 Cell。


In [3]:
MODEL_PATH = "baidu/ERNIE-4.5-0.3B-Paddle"

FD_HOST = "0.0.0.0"
FD_PORT = 8180
client = openai.Client(
    base_url=f"http://{FD_HOST}:{FD_PORT}/v1",
    api_key="null"
)
print(f"OpenAI client created, connecting to http://{FD_HOST}:{FD_PORT}/v1")


OpenAI client created, connecting to http://0.0.0.0:8180/v1


## 3. 游戏常量与配置

In [4]:
# 难度系数映射
DIFFICULTY_MULTIPLIERS = {1: 1.0, 2: 1.2, 3: 1.5, 4: 2.0, 5: 2.5}

# 难度对应的时间限制（秒）
DIFFICULTY_TIME_LIMITS = {1: 30, 2: 25, 3: 20, 4: 18, 5: 15}

# 难度对应的英文句子目标词数
DIFFICULTY_SENTENCE_LENGTH = {1: 10, 2: 20, 3: 30, 4: 40, 5: 50}

# 基础分
BASE_SCORE = 10

# 剩余时间奖励系数
TIME_BONUS_FACTOR = 0.5

# 连击奖励
STREAK_BONUS = 2

# 限时模式默认题数
TIMED_MODE_QUESTIONS = 10

# 题目生成最大重试次数
MAX_RETRIES = 2


## 4. 题目生成与解析

In [5]:
import random


def build_sentence_prompt(difficulty: int) -> str:
    """构建让大模型生成英文句子的 Prompt"""
    target_len = DIFFICULTY_SENTENCE_LENGTH.get(difficulty, 12)
    prompt = f"""请生成一句约 {target_len} 个词的英文句子。"""
    return prompt


def generate_sentence(difficulty: int) -> str:
    """调用大模型生成一句英文，带重试机制"""
    prompt = build_sentence_prompt(difficulty)
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model="null",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8,
                top_p=0.95,
                max_tokens=128,
                stream=False,
            )
            text = response.choices[0].message.content.strip()

            # 基本校验：非空且至少有4个词
            if text and len(text.split()) >= 4:
                return text
            print(f"generate_sentence: invalid output (attempt {attempt + 1}): {text[:80]}")
        except Exception as e:
            print(f"generate_sentence: error (attempt {attempt + 1}): {e}")
    return ""


def get_synonyms(word: str) -> list:
    """调用大模型获取一个单词的两个同义词/近义词，带重试机制"""
    import json
    prompt = f"""请为英文单词 "{word}" 提供 5 个同义词或近义词。以 json 格式返回
{{
    "synonyms": [
    ]
}}"""
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model="null",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.6,
                top_p=0.9,
                max_tokens=128,
                stream=False,
            )
            text = response.choices[0].message.content.strip()
            
            # 解析：从 JSON 中提取 synonyms 列表
            try:
                data = json.loads(text)
                syns = data.get("synonyms", [])
            except json.JSONDecodeError:
                # 如果 JSON 解析失败，尝试从文本中提取 JSON 部分
                match = re.search(r'\{[\s\S]*?\}', text)
                if match:
                    try:
                        data = json.loads(match.group())
                        syns = data.get("synonyms", [])
                    except json.JSONDecodeError:
                        syns = []
                else:
                    syns = []
            # 过滤掉和原词相同的，并确保是字符串
            syns = [w for w in syns if isinstance(w, str) and w.lower() != word.lower()]
            # 成功获取至少2个同义词则返回
            if len(syns) >= 2:
                return random.sample(syns, 2)
            print(f"get_synonyms('{word}'): not enough synonyms (got {len(syns)}, attempt {attempt + 1})")
        except Exception as e:
            print(f"get_synonyms('{word}'): error (attempt {attempt + 1}): {e}")
    return []


def pick_cloze_word(sentence: str) -> tuple:
    """
    从句子中选取一个适合作为完形填空空白的单词。
    返回 (word, index) 或 None。
    优先选择实词（名词、动词、形容词、副词），跳过冠词、介词、连词等。
    """
    # 常见虚词列表，这些词不适合作为完形填空的空白
    skip_words = {
        'a', 'an', 'the', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'am', 'do', 'does', 'did', 'have', 'has', 'had', 'having',
        'will', 'would', 'shall', 'should', 'can', 'could', 'may', 'might', 'must',
        'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him', 'her', 'us', 'them',
        'my', 'your', 'his', 'its', 'our', 'their', 'mine', 'yours', 'hers', 'ours',
        'this', 'that', 'these', 'those',
        'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'from', 'up', 'down',
        'out', 'off', 'over', 'under', 'about', 'into', 'through', 'after', 'before',
        'between', 'among', 'during', 'until', 'since',
        'and', 'or', 'but', 'not', 'nor', 'so', 'yet', 'if', 'then',
        'as', 'than', 'when', 'while', 'where', 'how', 'what', 'which', 'who',
        'very', 'too', 'also', 'just', 'only', 'even', 'still', 'already',
        'no', 'yes', 'all', 'each', 'every', 'both', 'any', 'some',
        'here', 'there', 'now', 'never', 'always', 'often',
    }

    # 分词并保留原始位置信息
    words = []
    for match in re.finditer(r"[a-zA-Z']+", sentence):
        words.append((match.group(), match.start(), match.end()))

    # 筛选适合的词（长度>=3的实词，不在虚词列表中）
    candidates = [(w, s, e) for w, s, e in words
                  if w.lower() not in skip_words and len(w) >= 3]

    if not candidates:
        # 退而求其次：选长度>=2的词
        candidates = [(w, s, e) for w, s, e in words
                      if w.lower() not in skip_words and len(w) >= 2]

    if not candidates:
        return None

    # 随机选一个
    chosen = random.choice(candidates)
    return chosen[0], chosen[1], chosen[2]


def build_cloze_question(sentence: str, word: str, start: int, end: int, synonyms: list) -> tuple:
    """
    构建完形填空题目。
    使用 start/end 精确替换被选中的单词，避免句子中有重复词时替换错位置。
    返回 (question_text, options_list, answer)
    """
    answer = word
    # 确保有3个选项
    options = [answer]
    for syn in synonyms:
        if syn.lower() != answer.lower() and syn not in options:
            options.append(syn)
    # 如果同义词不够，补充占位
    while len(options) < 3:
        options.append(f"option{len(options)}")
    options = options[:3]
    # 随机打乱选项顺序
    random.shuffle(options)

    # 使用位置信息精确替换被选中的单词
    options_str = ', '.join(options)
    question_text = sentence[:start] + f'[{options_str}]' + sentence[end:]

    return question_text, options, answer


def get_question(difficulty: int):
    """生成完形填空题目，带重试机制"""
    for attempt in range(MAX_RETRIES + 1):
        try:
            # Step 1: 生成英文句子
            sentence = generate_sentence(difficulty)
            
            # 去掉可能的引号包裹
            sentence = sentence.strip('"\'')
            if not sentence or len(sentence.split()) < 4:
                print(f"Generated sentence too short (attempt {attempt + 1}): {sentence[:80]}")
                continue

            # Step 2: 选取一个词作为空白
            pick_result = pick_cloze_word(sentence)
            
            if pick_result is None:
                print(f"No suitable cloze word found (attempt {attempt + 1})")
                continue
            word, start, end = pick_result

            # Step 3: 用大模型获取同义词
            synonyms = get_synonyms(word)
            
            if len(synonyms) < 2:
                print(f"Not enough synonyms for '{word}' (got {len(synonyms)}, attempt {attempt + 1})")
                continue

            # Step 4: 构建完形填空题
            question_text, options, answer = build_cloze_question(sentence, word, start, end, synonyms)
            return question_text, options, answer

        except Exception as e:
            print(f"Error generating question (attempt {attempt + 1}): {e}")
    return None


## 5. 游戏状态管理

In [6]:
class GameState:
    """管理游戏的全局状态"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.score = 0
        self.difficulty = 1
        self.streak = 0
        self.mode = "arcade"  # arcade / timed
        self.remaining_questions = TIMED_MODE_QUESTIONS
        self.total_correct = 0
        self.current_question = None   # (question_text, options, answer)
        self.game_active = False
        self.start_time = 0

    def get_time_limit(self) -> int:
        """获取当前难度对应的时间限制"""
        return DIFFICULTY_TIME_LIMITS.get(min(self.difficulty, 5), 15)

    def calculate_score(self, time_remaining: float) -> int:
        """计算单题得分"""
        diff_key = min(self.difficulty, 5)
        multiplier = DIFFICULTY_MULTIPLIERS[diff_key]
        time_bonus = time_remaining * TIME_BONUS_FACTOR
        streak_bonus = self.streak * STREAK_BONUS
        total = int(BASE_SCORE * multiplier + time_bonus + streak_bonus)
        return max(total, 1)


game = GameState()

---

## 6. 简单交互测试（Cell 逐行执行）

以下部分不使用 Gradio，通过逐个 Cell 执行来体验游戏流程。

### 6.1 选择游戏模式并初始化

In [7]:
# 选择模式："arcade" 或 "timed"
GAME_MODE = "arcade"

game.reset()
game.mode = GAME_MODE
game.game_active = True

mode_label = "Arcade (闯关模式)" if game.mode == "arcade" else "Timed (限时模式)"
print(f"Game started! Mode: {mode_label}")
print(f"Difficulty: {game.difficulty} | Time Limit: {game.get_time_limit()}s")

Game started! Mode: Arcade (闯关模式)
Difficulty: 1 | Time Limit: 30s


### 6.2 生成第一道题

**注意** 如果 Fastdeploy 还没有启动起来，会出现 `Connection error`，需要等待一段时间后重试

In [9]:
question_data = get_question(game.difficulty)

if question_data is None:
    print("Failed to generate question. Please re-run this cell.")
else:
    game.current_question = question_data
    game.start_time = time.time()

    q_text, options, answer = question_data
    display_text = re.sub(r'\[[^\]]+\]', '___', q_text)

    print(f"Difficulty: {game.difficulty} | Time Limit: {game.get_time_limit()}s")
    print(f"\nQuestion: {display_text}")
    print(f"\nOptions:")
    for i, opt in enumerate(options, 1):
        print(f"  {i}. {opt}")

Difficulty: 1 | Time Limit: 30s

Question: Snowflakes dance in the frost with efficiency and ___, as we wander the realm of winter's embrace.

Options:
  1. merit
  2. grace
  3. favor


### 6.3 答题

将 `YOUR_ANSWER` 修改为你选择的答案（必须是选项之一），然后运行此 Cell。

In [12]:
# 将下面的值改为你选择的答案
YOUR_ANSWER = "something"  # 例如: "years"

if not game.game_active or game.current_question is None:
    print("Game is not active. Please initialize first (run 6.1 and 6.2).")
elif YOUR_ANSWER == "":
    print("Please set YOUR_ANSWER to one of the options, then re-run this cell.")
else:
    q_text, options, correct_answer = game.current_question
    elapsed = time.time() - game.start_time
    time_remaining = max(game.get_time_limit() - elapsed, 0)

    is_correct = (YOUR_ANSWER == correct_answer)

    if is_correct:
        game.streak += 1
        earned = game.calculate_score(time_remaining)
        game.score += earned
        game.total_correct += 1
        print(f"Correct! +{earned} points (Time bonus: {time_remaining:.1f}s remaining)")
        if game.difficulty < 5:
            game.difficulty += 1
    else:
        print(f"Wrong! The correct answer is: {correct_answer}")
        if game.mode == "arcade":
            game.game_active = False
            print(f"\n--- GAME OVER ---")
            print(f"Final Score: {game.score} | Max Streak: {game.streak} | Difficulty Reached: {game.difficulty}")
        else:
            game.streak = 0

    # 限时模式检查剩余题数
    if game.mode == "timed" and game.game_active:
        game.remaining_questions -= 1
        if game.remaining_questions <= 0:
            game.game_active = False
            print(f"\n--- GAME OVER ---")
            print(f"Final Score: {game.score} | Correct: {game.total_correct}/{TIMED_MODE_QUESTIONS}")

    print(f"\nScore: {game.score} | Streak: {game.streak} | Difficulty: {game.difficulty}")

Correct! +17 points (Time bonus: 2.9s remaining)

Score: 35 | Streak: 2 | Difficulty: 3


### 6.4 生成下一道题

答对后运行此 Cell 生成下一题，然后回到 6.3 继续答题。

In [13]:
if not game.game_active:
    print("Game is over. Run 6.1 to start a new game.")
else:
    question_data = get_question(game.difficulty)

    if question_data is None:
        print("Failed to generate question. Please re-run this cell.")
    else:
        game.current_question = question_data
        game.start_time = time.time()

        q_text, options, answer = question_data
        display_text = re.sub(r'\[[^\]]+\]', '___', q_text)

        remaining_info = f" | Remaining: {game.remaining_questions}" if game.mode == "timed" else ""
        print(f"Difficulty: {game.difficulty} | Time Limit: {game.get_time_limit()}s{remaining_info}")
        print(f"\nQuestion: {display_text}")
        print(f"\nOptions:")
        for i, opt in enumerate(options, 1):
            print(f"  {i}. {opt}")

Difficulty: 3 | Time Limit: 20s

Question: The city's vibrant energy blossoms in the late afternoon, buzzing with life, a symphony of laughter and color. Time flies like a gentle breeze, ___ with it a blend of hope and anticipation.

Options:
  1. carrying up
  2. carrying
  3. carried


### 6.5 查看当前游戏状态

In [14]:
mode_label = "Arcade" if game.mode == "arcade" else "Timed"
status = "Active" if game.game_active else "Game Over"

print(f"Mode: {mode_label} | Status: {status}")
print(f"Score: {game.score} | Streak: {game.streak} | Difficulty: {game.difficulty}")
if game.mode == "timed":
    print(f"Remaining Questions: {game.remaining_questions} | Correct: {game.total_correct}")

Mode: Arcade | Status: Active
Score: 35 | Streak: 2 | Difficulty: 3


---

## 7. Gradio 应用

以下部分构建完整的 Gradio Web 界面。

In [ ]:
import gradio as gr

### 7.1 Gradio 回调函数

In [ ]:
def start_game(mode_choice):
    """初始化游戏"""
    game.reset()
    game.mode = mode_choice
    game.game_active = True

    question_data = get_question(game.difficulty)
    if question_data is None:
        return (
            "Failed to generate question. Please try again.",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            "", "", "", "", ""
        )

    game.current_question = question_data
    game.start_time = time.time()

    q_text, options, answer = question_data
    display_text = re.sub(r'\[[^\]]+\]', '___', q_text)

    mode_label = "Arcade Mode" if game.mode == "arcade" else "Timed Mode"
    info = f"Mode: {mode_label} | Difficulty: {game.difficulty} | Time Limit: {game.get_time_limit()}s"

    return (
        display_text,
        gr.update(value=options[0], visible=True),
        gr.update(value=options[1], visible=True),
        gr.update(value=options[2], visible=True),
        str(game.score), str(game.streak), str(game.difficulty),
        info, ""
    )


def handle_answer(selected_option):
    """处理用户答题"""
    if not game.game_active or game.current_question is None:
        return (
            "Game is not active. Please start a new game.",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            "", "", "", "", ""
        )

    q_text, options, correct_answer = game.current_question
    elapsed = time.time() - game.start_time
    time_remaining = max(game.get_time_limit() - elapsed, 0)

    is_correct = (selected_option == correct_answer)

    if is_correct:
        game.streak += 1
        earned = game.calculate_score(time_remaining)
        game.score += earned
        game.total_correct += 1
        feedback = f"Correct! +{earned} points (Time bonus: {time_remaining:.1f}s)"
        if game.difficulty < 5:
            game.difficulty += 1
    else:
        feedback = f"Wrong! The correct answer is: {correct_answer}"
        if game.mode == "arcade":
            game.game_active = False
            summary = (
                f"\n--- GAME OVER ---\n"
                f"Final Score: {game.score} | Max Streak: {game.streak} | "
                f"Difficulty Reached: {game.difficulty}"
            )
            return (
                feedback + summary,
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                str(game.score), str(game.streak), str(game.difficulty),
                "Game Over", ""
            )
        else:
            game.streak = 0

    if game.mode == "timed":
        game.remaining_questions -= 1
        if game.remaining_questions <= 0:
            game.game_active = False
            summary = (
                f"\n--- ALL QUESTIONS DONE ---\n"
                f"Final Score: {game.score} | Correct: {game.total_correct}/{TIMED_MODE_QUESTIONS}"
            )
            return (
                feedback + summary,
                gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
                str(game.score), str(game.streak), str(game.difficulty),
                "Game Over", ""
            )

    question_data = get_question(game.difficulty)
    if question_data is None:
        game.game_active = False
        return (
            feedback + "\nFailed to generate next question. Game ended.",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            str(game.score), str(game.streak), str(game.difficulty),
            "Error", ""
        )

    game.current_question = question_data
    game.start_time = time.time()

    new_q_text, new_options, _ = question_data
    display_text = re.sub(r'\[[^\]]+\]', '___', new_q_text)

    mode_label = "Arcade" if game.mode == "arcade" else "Timed"
    remaining_info = f" | Remaining: {game.remaining_questions}" if game.mode == "timed" else ""
    info = f"{mode_label} | Diff: {game.difficulty} | Time: {game.get_time_limit()}s{remaining_info}"

    return (
        display_text,
        gr.update(value=new_options[0], visible=True),
        gr.update(value=new_options[1], visible=True),
        gr.update(value=new_options[2], visible=True),
        str(game.score), str(game.streak), str(game.difficulty),
        info, feedback
    )

### 7.2 构建 Gradio 界面

In [ ]:
with gr.Blocks(title="English Cloze Challenge", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# English Cloze Challenge")
    gr.Markdown("Select a game mode and answer cloze questions to earn points!")

    with gr.Row():
        with gr.Column(scale=1):
            mode_radio = gr.Radio(
                choices=["arcade", "timed"],
                value="arcade",
                label="Game Mode",
                info="Arcade: game over on wrong answer | Timed: answer 10 questions"
            )
            start_btn = gr.Button("Start Game", variant="primary")

        with gr.Column(scale=2):
            info_text = gr.Textbox(label="Game Info", interactive=False)
            feedback_text = gr.Textbox(label="Feedback", interactive=False)

    gr.Markdown("---")

    question_display = gr.Textbox(label="Question", lines=3, interactive=False)

    with gr.Row():
        opt1_btn = gr.Button("Option 1", visible=False)
        opt2_btn = gr.Button("Option 2", visible=False)
        opt3_btn = gr.Button("Option 3", visible=False)

    gr.Markdown("---")

    with gr.Row():
        score_display = gr.Textbox(label="Score", value="0", interactive=False)
        streak_display = gr.Textbox(label="Streak", value="0", interactive=False)
        diff_display = gr.Textbox(label="Difficulty", value="1", interactive=False)

    # --- Event bindings ---

    start_btn.click(
        fn=start_game,
        inputs=[mode_radio],
        outputs=[
            question_display,
            opt1_btn, opt2_btn, opt3_btn,
            score_display, streak_display, diff_display,
            info_text, feedback_text
        ]
    )

    for btn in [opt1_btn, opt2_btn, opt3_btn]:
        btn.click(
            fn=handle_answer,
            inputs=[btn],
            outputs=[
                question_display,
                opt1_btn, opt2_btn, opt3_btn,
                score_display, streak_display, diff_display,
                info_text, feedback_text
            ]
        )

print("Gradio app built successfully!")

### 7.3 启动 Gradio 应用

In [ ]:
demo.launch(server_name="0.0.0.0", server_port=7860, share=True)